In [1]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

/home/roben/Codes/Practical-Deep-Learning/practicaldeeplearning/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("bentrevett/multi30k")

In [3]:
train_data = dataset["train"]
test_data = dataset["test"]

In [4]:
import re

def tokenize(text):
    text = text.lower()
    return re.findall(r"\w+|[^\w\s]", text)

In [5]:
def build_vocab(dataset, language):
    counter = Counter()

    for item in dataset:
        tokens = tokenize(item[language])
        counter.update(tokens)

    vocab = {
        "<unk>": 0,
        "<pad>": 1,
        "<sos>": 2,
        "<eos>": 3
    }

    for word in counter:
        vocab[word] = len(vocab)

    return vocab

In [6]:
en_vocab = build_vocab(train_data, "en")
de_vocab = build_vocab(train_data, "de")

In [7]:
def numericalize(tokens, vocab):

    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]

    return [vocab["<sos>"]] + ids + [vocab["<eos>"]]

In [8]:
class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, en_vocab, de_vocab):
        self.data = hf_dataset
        self.en_vocab = en_vocab
        self.de_vocab = de_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_tokens = tokenize(self.data[idx]["en"])
        trg_tokens = tokenize(self.data[idx]["de"])

        src_ids = numericalize(src_tokens, self.en_vocab)
        trg_ids = numericalize(trg_tokens, self.de_vocab)

        return torch.tensor(src_ids), torch.tensor(trg_ids)


In [9]:
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)

    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=en_vocab["<pad>"])
    trg_padded = pad_sequence(trg_batch, batch_first=True, padding_value=de_vocab["<pad>"])

    return src_padded, trg_padded

In [10]:
train_dataset = TranslationDataset(train_data, en_vocab, de_vocab)
test_dataset = TranslationDataset(test_data, en_vocab, de_vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

In [13]:
src_batch, trg_batch = next(iter(train_loader))
print(src_batch.shape, trg_batch.shape)

torch.Size([32, 23]) torch.Size([32, 22])


In [14]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=en_vocab["<pad>"])
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell

In [15]:
INPUT_DIM = len(en_vocab)
EMB_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2

encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS)

src_batch, trg_batch = next(iter(train_loader))
hidden, cell = encoder(src_batch)
print(hidden.shape, cell.shape)

torch.Size([2, 32, 512]) torch.Size([2, 32, 512])


In [20]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1, dropout=0.5):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=de_vocab["<pad>"])
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell):
        # input token: [batch] (a single token id per sequence in the batch)
        input_token = input_token.unsqueeze(1)
        embedded = self.dropout(self.embedding(input_token))

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

In [21]:
OUTPUT_DIM = len(de_vocab)

decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS)

# simulate first decoder step: input = <sos> token for every sequence in the batch
first_input = trg_batch[:, 0]  # should be all <sos> ids, shape [batch]
prediction, hidden, cell = decoder(first_input, hidden, cell)

print(prediction.shape)  # expect [batch, len(de_vocab)]

torch.Size([32, 17915])


In [22]:
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input_token = trg[:,0]

        for t in range(1, trg_len):
            prediction, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:,t,:] = prediction

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)

            input_token = trg[:,t] if teacher_force else top1

        return outputs


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS).to(device)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM, N_LAYERS).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

src_batch, trg_batch = next(iter(train_loader))
src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)

output = model(src_batch, trg_batch)
print(output.shape)  # expect [batch, trg_len, len(de_vocab)]

torch.Size([32, 25, 17915])


In [24]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=de_vocab["<pad>"])

In [25]:
# output: [batch, trg_len, vocab_size] -> [(batch*trg_len), vocab_size]
# trg:    [batch, trg_len]             -> [(batch*trg_len)]

output_dim = output.shape[-1]
output_flat = output[:, 1:, :].reshape(-1, output_dim)
trg_flat = trg_batch[:, 1:].reshape(-1)

loss = criterion(output_flat, trg_flat)
print(loss.item())

9.790167808532715


In [29]:
def train_epoch(model, dataloader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    for src_batch, trg_batch in dataloader:
        src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)

        optimizer.zero_grad()

        output = model(src_batch, trg_batch, teacher_forcing_ratio=0.5)

        output_dim = output.shape[-1]
        output_flat = output[:, 1:, :].reshape(-1, output_dim)
        trg_flat = trg_batch[:, 1:].reshape(-1)

        loss = criterion(output_flat, trg_flat)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss/ len(dataloader)


def evaluate_epoch(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src_batch, trg_batch in dataloader:
            src_batch, trg_batch = src_batch.to(device), trg_batch.to(device)

            output = model(src_batch, trg_batch, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output_flat = output[:, 1:, :].reshape(-1, output_dim)
            trg_flat = trg_batch[:, 1:].reshape(-1)

            loss = criterion(output_flat, trg_flat)
            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [30]:
import math
import time

N_EPOCHS = 10

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    test_loss = evaluate_epoch(model, test_loader, criterion)

    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1:02} | Time: {elapsed:.1f}s")
    print(f"\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}")
    print(f"\t Val. Loss: {test_loss:.3f} |  Val. PPL: {math.exp(test_loss):7.3f}")

KeyboardInterrupt: 